In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

In [2]:
spark = SparkSession.builder \
    .appName("ClickstreamConsumer") \
    .getOrCreate()


In [3]:
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("ip", StringType(), True),
    StructField("url", StringType(), True)
])

In [8]:
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "clickstream") \
    .option("failOnDataLoss", "false") \
    .option("startingOffsets", "earliest") \
    .load()


In [5]:
clickstream_df = kafka_df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

In [6]:
querry = clickstream_df.writeStream \
    .format("iceberg") \
    .option("checkpointLocation", "s3a://lakehouse/checkpoints/") \
    .outputMode("append") \
    .start("lakehouse.bronze.raw_clickstream")
    

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [7]:
querry.awaitTermination()

25/03/14 07:51:07 ERROR MicroBatchExecution: Query [id = d107c646-b414-4041-90cc-5bcdc362c17d, runId = c803a681-601f-4524-a866-443aea534b7c] terminated with error
java.lang.IllegalStateException: Partition clickstream-0's offset was changed from 201 to 8, some data may have been missed. 
Some data may have been lost because they are not available in Kafka any more; either the
 data was aged out by Kafka or the topic may have been deleted before all the data in the
 topic was processed. If you don't want your streaming query to fail on such cases, set the
 source option "failOnDataLoss" to "false".
    
	at org.apache.spark.sql.kafka010.KafkaMicroBatchStream.reportDataLoss(KafkaMicroBatchStream.scala:309)
	at org.apache.spark.sql.kafka010.KafkaMicroBatchStream.$anonfun$planInputPartitions$1(KafkaMicroBatchStream.scala:201)
	at org.apache.spark.sql.kafka010.KafkaMicroBatchStream.$anonfun$planInputPartitions$1$adapted(KafkaMicroBatchStream.scala:201)
	at org.apache.spark.sql.kafka010.Kafk

StreamingQueryException: [STREAM_FAILED] Query [id = d107c646-b414-4041-90cc-5bcdc362c17d, runId = c803a681-601f-4524-a866-443aea534b7c] terminated with exception: Partition clickstream-0's offset was changed from 201 to 8, some data may have been missed. 
Some data may have been lost because they are not available in Kafka any more; either the
 data was aged out by Kafka or the topic may have been deleted before all the data in the
 topic was processed. If you don't want your streaming query to fail on such cases, set the
 source option "failOnDataLoss" to "false".
    